# Notebook 4 — Técnicas Avanzadas de Prompting
**Clase 2: Ingeniería de Prompts** | IA Generativa

## Objetivos
- Dominar Zero-shot, Few-shot y Chain-of-Thought prompting
- Implementar `FewShotPromptTemplate` de LangChain
- Aplicar self-consistency para mejorar la fiabilidad
- Comparar técnicas con la misma tarea para elegir la mejor

---

In [5]:
!pip install langchain langchain-google-genai langchain-core

In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import (
    ChatPromptTemplate, FewShotChatMessagePromptTemplate,
    FewShotPromptTemplate, PromptTemplate
)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import SystemMessage, HumanMessage

from google.colab import userdata
API_KEY = userdata.get('GEMINI_API_KEY')

MODEL = "gemini-2.5-flash-lite"

llm = ChatGoogleGenerativeAI(model=MODEL, temperature=0.5, google_api_key=API_KEY)

print("✅ Entorno listo")

✅ Entorno listo


## 1. Zero-Shot Prompting

Pedimos al modelo que realice una tarea **sin ejemplos**. El modelo se basa únicamente en su conocimiento previo.

In [8]:
# Zero-shot: solo instrucción
TAREA = "Clasifica la siguiente reseña como POSITIVA, NEGATIVA o MIXTA."
RESENA = "La cámara del móvil es increíble, pero la batería dura muy poco y el precio es abusivo."

# Zero-shot básico
zero_shot_basico = ChatPromptTemplate.from_messages([
    ("system", TAREA),
    ("human", "{resena}")
])

# Zero-shot con instrucción más detallada
zero_shot_detallado = ChatPromptTemplate.from_messages([
    ("system", f"""{TAREA}

Criterios:
- POSITIVA: el cliente está satisfecho en general
- NEGATIVA: el cliente está insatisfecho en general
- MIXTA: hay tanto aspectos positivos como negativos

Responde SOLO con una palabra: POSITIVA, NEGATIVA o MIXTA."""),
    ("human", "{resena}")
])

print("=== ZERO-SHOT BÁSICO ===")
resp1 = (zero_shot_basico | llm | StrOutputParser()).invoke({"resena": RESENA})
print(resp1)

print("\n=== ZERO-SHOT DETALLADO ===")
resp2 = (zero_shot_detallado | llm | StrOutputParser()).invoke({"resena": RESENA})
print(resp2)

=== ZERO-SHOT BÁSICO ===
La reseña se clasifica como **MIXTA**.

**Justificación:**

*   **Positiva:** Menciona que "La cámara del móvil es increíble".
*   **Negativa:** Indica que "la batería dura muy poco" y "el precio es abusivo".

Al tener elementos positivos y negativos claros, la reseña es mixta.

=== ZERO-SHOT DETALLADO ===
MIXTA


## 2. Few-Shot Prompting

Proporcionamos **ejemplos input→output** para guiar al modelo. Especialmente útil cuando la tarea tiene un formato específico o es difícil de describir con palabras.

In [ ]:
# Ejemplos para el clasificador de reseñas
ejemplos_clasificacion = [
    {
        "resena": "Producto excelente, superó mis expectativas. Envío rapidísimo.",
        "clasificacion": "POSITIVA"
    },
    {
        "resena": "Una porquería. Se rompió al día siguiente. Servicio de atención al cliente nefasto.",
        "clasificacion": "NEGATIVA"
    },
    {
        "resena": "El diseño es bonito y el precio razonable, pero la calidad del material es mejorable.",
        "clasificacion": "MIXTA"
    },
    {
        "resena": "No es lo que esperaba. Las fotos engañan mucho.",
        "clasificacion": "NEGATIVA"
    },
]

# FewShotChatMessagePromptTemplate
ejemplo_prompt = ChatPromptTemplate.from_messages([
    ("human", "{resena}"),
    ("ai", "{clasificacion}")
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=ejemplo_prompt,
    examples=ejemplos_clasificacion
)

template_few_shot = ChatPromptTemplate.from_messages([
    ("system", "Eres un clasificador de sentimientos de reseñas. Responde SOLO con POSITIVA, NEGATIVA o MIXTA."),
    few_shot_prompt,  # Aquí van los ejemplos
    ("human", "{resena}")  # Y aquí la nueva pregunta
])

# Ver la estructura completa del prompt
print("Mensajes del prompt (con ejemplos):")
messages = template_few_shot.format_messages(resena="NUEVA RESEÑA")
for i, msg in enumerate(messages):
    print(f"  [{i}] {msg.__class__.__name__}: {msg.content[:60]}...")

In [ ]:
# Comparar zero-shot vs few-shot en casos difíciles
resenas_dificiles = [
    "No está mal para el precio que tiene.",           # Ambigua → MIXTA/POSITIVA
    "Exactamente lo que esperaba, ni más ni menos.",   # Ambigua → POSITIVA/NEUTRA
    "El producto OK pero el vendedor es un desastre.", # Mixta clara
]

cadena_zero = zero_shot_detallado | llm | StrOutputParser()
cadena_few  = template_few_shot  | llm | StrOutputParser()

print("Comparativa Zero-Shot vs Few-Shot en casos difíciles:\n")
print(f"{'Reseña':<50} {'Zero-Shot':<12} {'Few-Shot':<12}")
print("-" * 74)

for resena in resenas_dificiles:
    zs = cadena_zero.invoke({"resena": resena}).strip()
    fs = cadena_few.invoke({"resena": resena}).strip()
    print(f"'{resena[:47]}'  {zs:<12} {fs:<12}")

## 3. Chain-of-Thought (CoT) Prompting

Pedimos al modelo que **piense paso a paso** antes de dar la respuesta final. Mejora drásticamente el razonamiento en tareas complejas.

In [ ]:
# Problema de razonamiento
PROBLEMA = """
En una empresa hay 3 departamentos: Ventas (15 personas), Marketing (8 personas) y TI (12 personas).
El 40% de Ventas trabaja de forma remota.
El 75% de Marketing trabaja de forma remota.
El 50% de TI trabaja de forma remota.
¿Cuántas personas en total trabajan de forma presencial?
"""

# Sin CoT: respuesta directa
sin_cot = ChatPromptTemplate.from_messages([
    ("system", "Responde la pregunta directamente con el número."),
    ("human", "{problema}")
])

# Con CoT: pedimos razonamiento paso a paso
con_cot = ChatPromptTemplate.from_messages([
    ("system", """Resuelve el problema paso a paso.

Usa este formato:
PASO 1: [primer cálculo]
PASO 2: [segundo cálculo]
...
RESPUESTA FINAL: [resultado]"""),
    ("human", "{problema}")
])

print("=== SIN CHAIN-OF-THOUGHT ===")
resp_sin = (sin_cot | llm | StrOutputParser()).invoke({"problema": PROBLEMA})
print(resp_sin)

print("\n\n=== CON CHAIN-OF-THOUGHT ===")
resp_con = (con_cot | llm | StrOutputParser()).invoke({"problema": PROBLEMA})
print(resp_con)

In [ ]:
# Zero-Shot CoT: solo añadir "Piensa paso a paso"
# Esta es la forma más sencilla de activar el razonamiento
zero_shot_cot = ChatPromptTemplate.from_messages([
    ("human", "{problema}\n\nPiensa paso a paso antes de dar la respuesta.")
])

print("=== ZERO-SHOT CoT (solo 'piensa paso a paso') ===")
resp = (zero_shot_cot | llm | StrOutputParser()).invoke({"problema": PROBLEMA})
print(resp)

## 4. Few-Shot CoT — Combinar ejemplos + razonamiento

In [ ]:
# Ejemplos con razonamiento incluido
ejemplos_cot = [
    {
        "problema": "Una tienda tiene 50 camisetas. Vende el 30% el lunes y el 20% de las restantes el martes. ¿Cuántas le quedan?",
        "solucion": """RAZONAMIENTO:
- Lunes: 50 × 0.30 = 15 vendidas → quedan 50 - 15 = 35
- Martes: 35 × 0.20 = 7 vendidas → quedan 35 - 7 = 28
RESPUESTA: 28 camisetas"""
    },
    {
        "problema": "Si 5 trabajadores construyen una pared en 8 días, ¿cuánto tardarán 10 trabajadores?",
        "solucion": """RAZONAMIENTO:
- Trabajo total = 5 × 8 = 40 días-trabajador
- Con 10 trabajadores: 40 / 10 = 4 días
RESPUESTA: 4 días"""
    }
]

ejemplo_prompt_cot = ChatPromptTemplate.from_messages([
    ("human", "{problema}"),
    ("ai", "{solucion}")
])

few_shot_cot = FewShotChatMessagePromptTemplate(
    example_prompt=ejemplo_prompt_cot,
    examples=ejemplos_cot
)

template_final_cot = ChatPromptTemplate.from_messages([
    ("system", "Eres un matemático experto. Siempre muestras tu razonamiento antes de dar la respuesta."),
    few_shot_cot,
    ("human", "{problema}")
])

resp = (template_final_cot | llm | StrOutputParser()).invoke({"problema": PROBLEMA})
print("=== FEW-SHOT CoT ===")
print(resp)

## 5. Self-Consistency — Fiabilidad por votación

In [ ]:
# Generar múltiples respuestas y quedarse con la más frecuente
llm_variado = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0.8,  # Alta temperatura para variabilidad
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

def self_consistency(template, inputs: dict, n_muestras: int = 5, extractor=None):
    """
    Genera n respuestas y devuelve la más frecuente.
    extractor: función para extraer la respuesta final de cada output.
    """
    cadena = template | llm_variado | StrOutputParser()
    respuestas = [cadena.invoke(inputs) for _ in range(n_muestras)]

    # Extraer respuesta final si hay extractor
    if extractor:
        respuestas_finales = [extractor(r) for r in respuestas]
    else:
        respuestas_finales = [r.strip() for r in respuestas]

    # Votación
    conteo = Counter(respuestas_finales)
    ganadora = conteo.most_common(1)[0][0]

    return {
        "respuesta_consenso": ganadora,
        "votos": dict(conteo),
        "confianza": conteo[ganadora] / n_muestras,
        "todas_respuestas": respuestas
    }


# Template de clasificación
template_clasificacion_simple = ChatPromptTemplate.from_messages([
    ("system", "Clasifica el sentimiento. Responde ÚNICAMENTE con: POSITIVA, NEGATIVA o MIXTA."),
    ("human", "{resena}")
])

resena_ambigua = "No es el mejor producto que he comprado, pero tampoco el peor."

print(f"Reseña: '{resena_ambigua}'\n")
print(f"Generando {5} respuestas independientes...\n")

resultado = self_consistency(
    template_clasificacion_simple,
    {"resena": resena_ambigua},
    n_muestras=5
)

print(f"Votos: {resultado['votos']}")
print(f"Respuesta por consenso: {resultado['respuesta_consenso']}")
print(f"Confianza: {resultado['confianza']*100:.0f}%")

In [ ]:
# Self-consistency con razonamiento numérico
def extraer_numero(texto: str) -> str:
    """Extrae el número final de una respuesta CoT."""
    import re
    lineas = texto.strip().split("\n")
    for linea in reversed(lineas):
        numeros = re.findall(r'\b\d+\b', linea)
        if numeros:
            return numeros[-1]
    return texto.strip()

problema_numerico = """
Un tren sale de Madrid a las 8:00 a 120 km/h.
Otro sale de Barcelona a las 9:00 a 100 km/h en dirección contraria.
La distancia entre ciudades es de 600 km.
¿A qué hora se cruzan? Da la respuesta como número de minutos desde las 8:00.
"""

resultado_num = self_consistency(
    template_final_cot,
    {"problema": problema_numerico},
    n_muestras=5,
    extractor=extraer_numero
)

print("Problema del tren — Self-Consistency:")
print(f"Votos por respuesta: {resultado_num['votos']}")
print(f"Respuesta consenso: {resultado_num['respuesta_consenso']} minutos")
print(f"Confianza: {resultado_num['confianza']*100:.0f}%")

## 6. Otras técnicas importantes

In [ ]:
# Técnica: Role Prompting (hacer que el modelo adopte un rol específico)
roles = [
    ("CEO",        "Eres el CEO de una empresa de IA. Responde desde una perspectiva estratégica y de negocio."),
    ("Ingeniero",  "Eres un ingeniero de software senior. Responde con detalle técnico e implementación práctica."),
    ("Crítico",    "Eres un periodista crítico especializado en tecnología. Señala siempre los riesgos y limitaciones."),
]

PREGUNTA_ROL = "¿Cuál es el impacto de los LLMs en el mercado laboral?"

for rol, system in roles:
    template_rol = ChatPromptTemplate.from_messages([
        ("system", system),
        ("human", "{pregunta}")
    ])
    resp = (template_rol | llm | StrOutputParser()).invoke({"pregunta": PREGUNTA_ROL})
    print(f"\n🎭 [{rol}]")
    print(resp[:400])

In [ ]:
# Técnica: Negative Prompting (decir lo que NO debe hacer)
template_sin_negativos = ChatPromptTemplate.from_messages([
    ("system", """Eres un redactor de contenido de marketing.

NO hagas lo siguiente:
- No uses superlativos vacíos como 'el mejor', 'increíble', 'revolucionario'
- No uses jerga de startup como 'disruptivo', 'game-changer', 'paradigma'
- No incluyas frases hechas como 'en el mundo actual' o 'en el panorama actual'
- No uses más de 100 palabras

SÍ haz lo siguiente:
- Usa datos concretos cuando sea posible
- Habla de beneficios específicos para el usuario
- Usa un tono directo y honesto"""),
    ("human", "Escribe un párrafo de presentación para {producto}")
])

resp = (template_sin_negativos | llm | StrOutputParser()).invoke({
    "producto": "una plataforma de análisis de sentimientos con IA"
})
print("=== NEGATIVE PROMPTING ===")
print(resp)

## 7. Tabla comparativa de técnicas

In [ ]:
# Comparativa final con la misma tarea difícil
TAREA_DIFICIL = """
Clasifica esta reseña de producto y proporciona una puntuación del 1-5:

"El producto en sí es excelente pero tardó 3 semanas en llegar (pedí envío express),
el embalaje llegó dañado aunque el interior estaba bien, y el servicio de atención
al cliente no respondió mis emails. Sin embargo, la calidad supera con creces las
expectativas y lo volvería a comprar si el proceso de compra fuera mejor."
"""

tecnicas = {
    "Zero-Shot": ChatPromptTemplate.from_messages([
        ("system", "Clasifica la reseña (POSITIVA/NEGATIVA/MIXTA) y da una puntuación 1-5. Formato: CLASIFICACIÓN | PUNTUACIÓN"),
        ("human", "{tarea}")
    ]),
    "Zero-Shot CoT": ChatPromptTemplate.from_messages([
        ("system", "Analiza la reseña paso a paso y clasifícala (POSITIVA/NEGATIVA/MIXTA) con puntuación 1-5."),
        ("human", "{tarea}\n\nPiensa paso a paso considerando todos los aspectos mencionados.")
    ]),
    "Role Prompting": ChatPromptTemplate.from_messages([
        ("system", "Eres un analista de experiencia de cliente con 10 años de experiencia evaluando reseñas para empresas de e-commerce. Clasifica (POSITIVA/NEGATIVA/MIXTA) con puntuación 1-5 y justificación."),
        ("human", "{tarea}")
    ]),
}

print("COMPARATIVA DE TÉCNICAS")
print("=" * 60)
for nombre, template in tecnicas.items():
    print(f"\n🔹 {nombre}")
    print("-" * 40)
    resp = (template | llm | StrOutputParser()).invoke({"tarea": TAREA_DIFICIL})
    print(resp[:500])

## 8. ¿Cuándo usar cada técnica?

| Técnica | Cuándo usarla | Ventajas | Desventajas |
|---|---|---|---|
| **Zero-Shot** | Tareas claras y bien definidas | Rápido, pocos tokens | Menos preciso en tareas complejas |
| **Few-Shot** | Formato específico, clasificaciones | Alta precisión, consistente | Más tokens, requiere ejemplos |
| **CoT** | Razonamiento, matemáticas, lógica | Mucho más preciso | Respuestas largas, más lento |
| **Few-Shot CoT** | Problemas complejos con formato | Lo mejor de ambos mundos | Muchos tokens |
| **Self-Consistency** | Decisiones críticas, baja tolerancia al error | Mayor fiabilidad | N veces el coste |
| **Role Prompting** | Necesitas perspectiva específica | Cambia el punto de vista | Puede sobreactuar |
| **Negative Prompting** | Evitar comportamientos conocidos | Control fino | Puede ignorar algunas restricciones |

**➡️ Siguiente:** Notebook 5 — Output Parsers y structured outputs

### 🏋️ Ejercicios
1. Diseña un Few-Shot prompt para clasificar correos de soporte técnico en: `bug`, `feature_request`, `pregunta`, `queja`
2. Aplica Chain-of-Thought para resolver: "Un e-commerce tiene una tasa de conversión del 2.5% con 10.000 visitas diarias. Si mejoran la conversión a 3.2%, ¿cuántos pedidos más harán en un mes?"
3. Usa self-consistency con 7 muestras para responder esa misma pregunta. ¿Coincide la respuesta?